# Quickstart: Your First Bayesian VAR

This tutorial walks you through fitting a Bayesian VAR model to macroeconomic data using Litterman.

In [ ]:
import numpy as np
import pandas as pd

from litterman import VAR, VARData, select_lag_order
from litterman.samplers import NUTSSampler

## Create some synthetic data

We'll simulate a simple VAR(1) process with two variables.

In [ ]:
rng = np.random.default_rng(42)
T = 200
y = np.zeros((T, 2))
for t in range(1, T):
    y[t] = 0.5 * y[t - 1] + rng.standard_normal(2) * 0.1

index = pd.date_range("2000-01-01", periods=T, freq="QS")
data = VARData(endog=y, endog_names=["gdp", "inflation"], index=index)
data

## Select lag order

Use information criteria to find the optimal lag length.

In [ ]:
ic = select_lag_order(data, max_lags=8)
print(f"AIC: {ic.aic}, BIC: {ic.bic}, HQ: {ic.hq}")
ic.summary()

## Specify and estimate the model

Create a VAR specification and fit it. We use a small number of draws here for speed.

In [ ]:
spec = VAR(lags="bic", prior="minnesota")
sampler = NUTSSampler(draws=500, tune=500, chains=2, cores=1, random_seed=42)
fitted = spec.fit(data, sampler=sampler)
fitted

## Inspect the posterior

Access the ArviZ InferenceData directly for diagnostics.

In [ ]:
import arviz as az

az.summary(fitted.idata, var_names=["B", "intercept"])